## 1 Data Preparation 

The 1_create_reference.ipynb notebook serves as the foundational data preparation step for our sequencing alignment pipeline. Because we are sequencing a custom pCRISPRi dual-guide RNA library, standard genomic references cannot be used. Instead, we must computationally reconstruct the exact amplicon sequences we expect to see in our sequencing reads and build custom databases from them.

Specifically, this script takes our raw sgRNA candidate sequences and flanks them with their respective upstream and downstream vector sequences to generate full-length reference amplicons.

## Key Steps in the Notebook:
- Importing Raw Sequences: The script reads in two Excel files: sgRNA_candidates.xlsx (containing the variable protospacer sequences) and addition_sequences.xlsx (containing the constant flanking sequences).

- Constructing the Amplicons: It computationally builds two distinct sets of reference sequences to match our cloning strategy:

    - 171 bp Amplicon (Protospacer A/C): Concatenates the mU6_protospacerC sequence, the variable protospacer_A, and the protospacerC_704 sequence.

    - 164 bp Amplicon (Protospacer B/D): Concatenates the LKO_protospacerD sequence, the variable protospacer_B, and the protospacerD_705 sequence.

- Generating FASTA Files: It exports these fully constructed sequences into combined FASTA files (pCRISPRi_dual_protospacer_AC_171.fa and pCRISPRi_dual_protospacer_BD_164.fa).

- Building Alignment Databases: Using the command line tool makeblastdb, the script converts these FASTA files into formatted nucleotide databases.

- Creating Individual References: Finally, it creates dedicated directories containing individual FASTA files for every single sgRNA candidate.

## Why is this necessary?
By generating these tailored reference files, we provide the exact "targets" that our downstream alignment tools (like bwa mem and samtools used in the bash scripts) need to accurately map the NGS reads. This allows us to calculate read coverage specifically across the protospacer loci to assess the distribution and potential bias of our plasmid pool and transduced gDNA.

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO
from Bio.Seq import Seq
import glob, os
import plotly
import plotly.graph_objects as go


In [17]:
def createfasta(input_df, index_col="index",fasta_col="protospacer_53",output_file="./database.fa"):
    """""reads in index file as the name and the fasta column and outputs into a fasta file"""""
    columns=[index_col,fasta_col]
    # input_df=pd.read_excel(input_file,header=None, names=columns)
    fasta=[]
    ##Create a fasta file for all of the files 
    for index,row in input_df.iterrows(): 
        fasta.append((">"+row[index_col]))
        fasta.append((row[fasta_col]))



    fasta_file = pd.DataFrame(fasta)
    fasta_file.to_csv("tst.fa",sep="\t",index=None, header=None)
    cmd = 'cat tst.fa | sed "s/ //g" > '+output_file
    # cmd = 'sed "s/[[:space:]]*$//g"  tst.fa > '+output_file
    os.system(cmd)
    # fasta_file.to_csv(output_file,sep="\t",index=None, header=None)


def sanger_database(input_db_fasta):
    """""Input a folder and folder location with database location"""
    makedb_cmd="makeblastdb -dbtype nucl -in "+input_db_fasta
    #read in the different protospacers 
    print(makedb_cmd) 
    os.system(makedb_cmd)


def prepend_line(file_name, line, dummy_file="tmp.fa"):
    """ Insert given string as a new line at the beginning of a file """
    # open original file in read mode and dummy file in write mode
    with open(file_name, 'r') as read_obj, open(dummy_file, 'w') as write_obj:
        # Write given line to the dummy file
        write_obj.write(line + '\n')
        # Read lines from original file one by one and append them to the dummy file
        for line in read_obj:
            write_obj.write(line)
    # remove original file
    # os.remove(file_name)
    # Rename dummy file as the original file
    # os.rename(dummy_file, file_name)


def sanger_blast(input_folder, folder_location, database_loc):
    """""Input a folder and folder location with database location"""
        #read in the different protospacers 

    ##first create a list of all the sequence files in a paticular path
    list_output=[]
    sanger_file_list=[]
    for file in os.listdir(folder_location):
        if file.endswith(".seq"):
            sanger_file_list.append(os.path.join("./",input_folder, file))

    #Edit each sanger sequencing file to change from seq to fasta file
    for file_name in sanger_file_list:
        ##Edit the sanger sequence to make it a fasta file
        name=(file_name.split("/")[2].split(".")[0])
    
        addline=">"+name
        prepend_line(file_name=file_name, line=addline)
        blastn_command='blastn -query tmp.fa -db '+database_loc+' -outfmt "10 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore" -out results.out -task blastn-short'
        os.system(blastn_command)
        # print(blastn_command)
        results=pd.read_csv("results.out", names=["qseqid","sseqid" ,"pident", "length" , "mismatch" , "gapopen" ,"qstart" ,"qend" ,"sstart" , "send", "evalue", "bitscore"])
        
        
        blastn_command='blastn -query tmp.fa -db '+database_loc+' -out results_display.out -task blastn-short'
        os.system(blastn_command)
        # print(blastn_command)
        list_output.append(results)

        append_commandline="cat results_display.out >> results_display_total.out"
        os.system(append_commandline)


    merged = pd.concat(list_output)
    return merged


def df_to_plotly(df):
    return {'z': df.values.tolist(),
            'x': df.columns.tolist(),
            'y': df.index.tolist() }
            
def plotly_create(input_dataframe,  save_file, value_column, index1="qseqid", index2="sseqid", title="Heatmap"):
    input_dataframe[value_column]=input_dataframe[value_column].astype(int)
    pivot_input_dataframe=pd.pivot_table(input_dataframe, values=value_column, index=index1, columns=index2)
    fig = go.Figure(
        data=go.Heatmap(df_to_plotly(pivot_input_dataframe)),
        layout=go.Layout(
    title=go.layout.Title(text=title)
        )
    )
    fig.show()
    plotly.offline.plot(fig, filename = save_file, auto_open=False)


def barchart_protospacer_count(input_df, startswith="RW", title="Bar Chart Showing the Number of Protospacers",savefigname="histogram"):
    input_df_prot=input_df[(input_df["sseqid"].str.startswith(startswith)) ]
    input_df_prot_len=input_df_prot[(input_df_prot["length"]>7)]
    input_df_prot_len_tst=input_df_prot_len[['qseqid','sseqid']].groupby("qseqid").count()
    # sns.histplot(input_df_prot_len_tst["sseqid"])
    input_df_prot_len_tst["sseqid"].value_counts().astype(int).sort_index(ascending=True).plot(kind = 'bar')
    plt.title(title)
    plt.xlabel("Number of Protospacers found in Sequence")
    save="./plotly/"+savefigname+".png"
    plt.savefig(save)

def createindividualfasta(input_df, index_col="index",fasta_col="protospacer_53",output_db_folder="database"):
    #open folder and 
    cmd="mkdir -p "+output_db_folder
    os.system(cmd)

    columns=[index_col,fasta_col]
    # input_df=pd.read_excel(input_file,header=None, names=columns, index_col=None)
    # print(input_df.head())
    input_df[fasta_col]=input_df[fasta_col].str.upper()
    ##Create a fasta file for all of the files 
    count=1
    for index,row in input_df.iterrows(): 
        fasta=[]
        fasta.append((">"+row[index_col]))
        fasta.append((row[fasta_col]))
        fasta_file = pd.DataFrame(fasta)
        fasta_file.to_csv("tst.fa",sep="\t",index=None, header=None)
        output_file=output_db_folder+"/"+str(count)+".fa"
        count=count+1

        cmd = 'cat tst.fa | sed "s/ //g" > '+output_file
        os.system(cmd)


In [18]:
#read in the different protospacers and the sequences before and after for 164bp 171bp
sequences=pd.read_excel("../files/addition_sequences.xlsx", skiprows=1, names=["sequences"])

#read in 164bp needs t have 
sgrna_candidates=pd.read_excel("../files/sgRNA_candidates.xlsx", index_col=0)

###need to create two fasta files one for the 164bp and one for the 171bp but have to treat and and b seperately
##start - 171bp with mU6_protospacerC protospacer_A protospacerC_704
sgrna_candidates_171 = sgrna_candidates[["oligo_ID","protospacer_A"]]
sgrna_candidates_171["sequence"]=sequences.loc["mU6_protospacerC","sequences"]+sgrna_candidates_171["protospacer_A"]+sequences.loc["protospacerC_704","sequences"]
sgrna_candidates_171["oligo_ID"]=sgrna_candidates_171["oligo_ID"]+"_171"
##second  - 164bp with LKO_protospacerD protospacer_B protospacerD_705
sgrna_candidates_164 = sgrna_candidates[["oligo_ID","protospacer_B"]]
sgrna_candidates_164["sequence"]=sequences.loc["LKO_protospacerD","sequences"]+sgrna_candidates_164["protospacer_B"]+sequences.loc["protospacerD_705","sequences"]    
sgrna_candidates_164["oligo_ID"]=sgrna_candidates_164["oligo_ID"]+"_164"


/var/folders/r_/bqnt2f_d6919cz7v1yzx2ykc0000gn/T/ipykernel_52380/2118405899.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sgrna_candidates_171["sequence"]=sequences.loc["mU6_protospacerC","sequences"]+sgrna_candidates_171["protospacer_A"]+sequences.loc["protospacerC_704","sequences"]
/var/folders/r_/bqnt2f_d6919cz7v1yzx2ykc0000gn/T/ipykernel_52380/2118405899.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sgrna_candidates_171["oligo_ID"]=sgrna_candidates_171["oligo_ID"]+"_171"
/var/folders/r_/bq

In [19]:
createfasta(sgrna_candidates_164,index_col="oligo_ID",fasta_col="sequence",output_file="../files/pCRISPRi_dual_protospacer_AC_171.fa")
createfasta(sgrna_candidates_171,index_col="oligo_ID",fasta_col="sequence",output_file="../files/pCRISPRi_dual_protospacer_BD_164.fa")


sanger_database("../files/pCRISPRi_dual_protospacer_AC_171.fa")
sanger_database("../files/pCRISPRi_dual_protospacer_BD_164.fa")

makeblastdb -dbtype nucl -in ../files/pCRISPRi_dual_protospacer_AC_171.fa


Building a new DB, current time: 02/20/2026 16:32:44
New DB name:   /Users/helenking/Library/CloudStorage/OneDrive-UNSW/sanger-sequencing-alignment/files/pCRISPRi_dual_protospacer_AC_171.fa
New DB title:  ../files/pCRISPRi_dual_protospacer_AC_171.fa
Sequence type: Nucleotide
Deleted existing Nucleotide BLAST database named /Users/helenking/Library/CloudStorage/OneDrive-UNSW/sanger-sequencing-alignment/files/pCRISPRi_dual_protospacer_AC_171.fa
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 70 sequences in 0.00406599 seconds.


makeblastdb -dbtype nucl -in ../files/pCRISPRi_dual_protospacer_BD_164.fa


Building a new DB, current time: 02/20/2026 16:32:44
New DB name:   /Users/helenking/Library/CloudStorage/OneDrive-UNSW/sanger-sequencing-alignment/files/pCRISPRi_dual_protospacer_BD_164.fa
New DB title:  ../files/pCRISPRi_dual_protospacer_BD_164.fa
Sequence type: Nucleotide
Deleted

In [20]:
sgrna_candidates_164

,oligo_ID,protospacer_B,sequence
1,sgTP53_1_AB_164,AAGTCTAGAGCCACCGTCCA,GACTATCATATGCTTACCGTAACTTGAAAGTATTTCGATTTCTTGG...
2,sgTP53_2_CD_164,TGGGAGCGTGCTTTCCACGA,GACTATCATATGCTTACCGTAACTTGAAAGTATTTCGATTTCTTGG...
3,sgTOMM22_1_AB_164,CGGCGGCAGCCATGACTGTA,GACTATCATATGCTTACCGTAACTTGAAAGTATTTCGATTTCTTGG...
4,sgTOMM22_2_CD_164,CCTTTCGGGAGCAATTCGTC,GACTATCATATGCTTACCGTAACTTGAAAGTATTTCGATTTCTTGG...
5,sgOTX2_1_AB_164,CTGAGGCCGGTCCCGCTCTC,GACTATCATATGCTTACCGTAACTTGAAAGTATTTCGATTTCTTGG...
...,...,...,...
66,sg_negcntrl_7_164,GTAAACCGCCGTGAACAAGG,GACTATCATATGCTTACCGTAACTTGAAAGTATTTCGATTTCTTGG...
67,sg_negcntrl_8_164,GGGTCGATGGAGTTCGCCCG,GACTATCATATGCTTACCGTAACTTGAAAGTATTTCGATTTCTTGG...
68,sg_negcntrl_9_164,GTTAAACGCCGCTATTTTGA,GACTATCATATGCTTACCGTAACTTGAAAGTATTTCGATTTCTTGG...
69,sg_negcntrl_19_164,GTCTTCGGCCCCTCCCATCG,GACTATCATATGCTTACCGTAACTTGAAAGTATTTCGATTTCTTGG...


In [21]:
#create a folder with one fasta file for every for the 

createindividualfasta(sgrna_candidates_164,index_col="oligo_ID",fasta_col="sequence",output_db_folder="../files/pCRISPRi_dual_protospacer_AC_171")
createindividualfasta(sgrna_candidates_171,index_col="oligo_ID",fasta_col="sequence",output_db_folder="../files/pCRISPRi_dual_protospacer_BD_164")


/var/folders/r_/bqnt2f_d6919cz7v1yzx2ykc0000gn/T/ipykernel_52380/1985519054.py:118: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  input_df[fasta_col]=input_df[fasta_col].str.upper()


In [ ]:
# createindividualfasta(input_file="../files/pCRISPRi_dual_TP53AP2_photospacer_AC_171.xlsx", output_db_folder="../pCRISPRi_dual_TP53AP2_photospacer_AC_171")
# createindividualfasta(input_file="../files/pCRISPRi_dual_TP53AP2_photospacer_BD_164.xlsx", output_db_folder="../pCRISPRi_dual_TP53AP2_photospacer_BD_164")
# ###Need to edit the sequences so that they include the protospacer sequences into the area surrounded by
# ##This is to form a sample specific bam file 
# createfasta(input_file="../files/pCRISPRi_dual_TP53AP2_photospacer_AC_171.xlsx", output_file="../files/pCRISPRi_dual_TP53AP2_photospacer_AC_171.fa")
# createfasta(input_file="../files/pCRISPRi_dual_TP53AP2_photospacer_BD_164.xlsx", output_file="../files/pCRISPRi_dual_TP53AP2_photospacer_BD_164.fa")